In [1]:
import pandas as pd
import mysql.connector

In [2]:
df = pd.read_csv('border_cleaned.csv')

In [3]:
df.head(3)

,port_name,state,port_code,border,date,measure,value,latitude,longitude,point
0,Jackman,Maine,104,US-Canada Border,2024-01-01,Trucks,6556,45.806,-70.397,POINT (-70.396722 45.805661)
1,Porthill,Idaho,3308,US-Canada Border,2024-04-01,Trucks,98,49.000,-116.499,POINT (-116.49925 48.999861)
2,San Luis,Arizona,2608,US-Mexico Border,2024-04-01,Buses,10,32.485,-114.782,POINT (-114.7822222 32.485)


In [5]:
connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password='password'   
    database='border_project'
)

cursor = connection.cursor()



In [11]:

df = df.where(pd.notnull(df), None)


In [13]:
df.dropna(subset=['state', 'latitude', 'longitude', 'point'], inplace=True)


In [15]:
df.replace('nan', None, inplace=True)


In [16]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df.replace('', None, inplace=True)


In [17]:
df.isnull().sum()


port_name    0
state        0
port_code    0
border       0
date         0
measure      0
value        0
latitude     0
longitude    0
point        0
dtype: int64

In [18]:
insert_query = """
INSERT INTO border_crossing
(port_name, state, port_code, border, date, measure, value, latitude, longitude, point)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

for index, row in df.iterrows():
    data = (
        row['port_name'],
        row['state'],
        int(row['port_code']),
        row['border'],
        row['date'],
        row['measure'],
        int(row['value']),
        float(row['latitude']),
        float(row['longitude']),
        row['point']
    )
    cursor.execute(insert_query, data)

connection.commit()